In [81]:
import numpy as np
import matplotlib.pyplot as plt
import screwCalculus as sc

"""
defining the problem: we need to compute the temperature transient in a pin holder.
The heat is generated from the walls of a cylindrcal heating chamber. The heating chamber heats the air inside it.
The air heats the steel pin holder. The steel pin holder is partially inserted into the heating chamber.
We can control the heating power of the heating chamber.
We want to compute the temperature transient in the pin holder.

Afterwards we want to minimize the time it takes for the pin holder to reach the steady state temperature.
"""

# Parameters
R = 0.1  # Radius of the heating chamber (m)
c_air = 1005 # Specific heat of air (J/kg/K)
rho_air = 1.225 # Density of air (kg/m^3)
k_air = 0.0257 # Thermal conductivity of air (W/m/K)
alpha_air = k_air/(rho_air*c_air) # Thermal diffusivity of air (m^2/s)

c_steel = 500 # Specific heat of steel (J/kg/K)
rho_steel = 7800 # Density of steel (kg/m^3)
k_steel = 50 # Thermal conductivity of steel (W/m/K)
h_steel_air = 12.5 # Heat transfer coefficient between steel and air (W/m^2/K)
alpha_steel = k_steel/(rho_steel*c_steel) # Thermal diffusivity of steel (m^2/s)

L = 0.15 # lenght of the pin holder (m)
L_inserted = 0.07 # lenght of the holder inserted into the heating chamber
L_ext = L-L_inserted
R_holder = 0.05 # radius of the pin holder (m)
A_holder_inserted = 2*np.pi*R_holder*L_inserted + np.pi*R_holder**2 # surface area of the inserted part of the pin holder
A_holder_ext = 2*np.pi*R_holder*L_ext + np.pi*R_holder**2           # surface area of the external part of the pin holder

H = 0.1 # height of the heating chamber (m)
A = 2*np.pi*R*H # heat flux area of the heating chamber


N_r = 10 # Number of radial nodes of the heating chamber
N_l = 10 # Number of longitudinal nodes of the pin holder
N_l_inserted = int(N_l*L_inserted/L) # Number of longitudinal nodes of the inserted part of the pin holder
N_l_ext = N_l - N_l_inserted

delta_r = R/N_r
delta_x = L/N_l
delta_vol = np.pi*R_holder**2*delta_x
# Define initial conditions
T0 = 300 # initial temperature of the pin holder
T_chamber0 = 300 # initial temperature of the heating chamber


In [82]:

from casadi import *
T = MX.sym('T', 1, 1) # generic temperature
T_prev = MX.sym('T_prev', 1, 1) # generic temperature at the previous spatial node
T_next = MX.sym('T_next', 1, 1) # generic temperature at the next spatial node
r = MX.sym('r', 1, 1) # generic radius
r_prev = MX.sym('r_prev', 1, 1) # generic radius at the previous spatial node   
r_next = MX.sym('r_next', 1, 1) # generic radius at the next spatial node
T_chamber = MX.sym('T_chamber', 1, 1) # temperature of the heating chamber

# control parameter
q_exchanged = MX.sym('Q_exchanged', 1, 1) # heat exchanged between the air in the heating chamber and the steel pin holder through convection
Q = MX.sym('Q', 1, 1) # heating power of the heating chamber
q = Q/A # heat flux from the heating chamber


# Define the spatial discretized temperature ODE
T_dot = alpha_steel*(T_next - 2*T + T_prev)/(delta_x)**2 + q_exchanged/(rho_steel*c_steel)
T_dot_fun_holder = Function('T_dot', [T,  q_exchanged, vertcat(T_prev, T_next)], [T_dot])

# Define the spatial discretized temperature ODE in cylindrical coordinates

T_dot = alpha_air/delta_r**2/r*(r_next*T_next - r_next*T - r*T + r*T_prev) + q/(rho_air*c_air)
T_dot_fun_chamber = Function('T_dot', [T, Q, vertcat(T_prev, T_next, r, r_prev, r_next)], [T_dot])


In [ ]:
T_dot_fun_holder(horzcat(350, 360), horzcat(100, 120), vertcat(horzcat(340, 360), horzcat(340, 360)))

print(vertcat(horzcat(340, 360), horzcat(340, 360)))

In [141]:
N_t = 5000 # Number of time steps

# Discretized temperature ODE
T = MX.sym('T', N_l, 1) # temperature of the pin holder
T_prev = vertcat(T[0], T[:-1]) # temperature of the pin holder at the previous spatial node
T_next = vertcat(T[1:], T[-1]) # temperature of the pin holder at the next spatial node

T_chamber = MX.sym('T_chamber', N_r, 1) # temperature of the heating chamber
T_chamber_prev = vertcat(T_chamber[0], T_chamber[:-1]) # temperature of the heating chamber at the previous spatial node
T_chamber_next = vertcat(T_chamber[1:], T_chamber[-1]) # temperature of the heating chamber at the next spatial node


r = MX.sym('r', N_r, 1) # radius of the heating chamber 
r_prev = vertcat(r[0], r[:-1]) # radius of the heating chamber at the previous spatial node
r_next = vertcat(r[1:], r[-1]) # radius of the heating chamber at the next spatial node

Q = MX.sym('Q', N_t, 1) # heating power of the heating chamber

T_ambient = 300 # ambient temperature

T_holder = T0*np.ones((N_l, 1))
T_chamber = T_chamber0*np.ones((N_r, 1))
r_num = np.linspace(0, R, N_r).reshape(-1, 1)

time_horizon = 3600*1.5 # s
time = np.linspace(0, time_horizon, N_t)
delta_t = time[-1]-time[-2] # time
t = 0
T_set = 500 + T_ambient
T_set_vec = np.ones((N_t, 1))
id = time < 3600*1.3
T_set_vec[id] = T_set
T_set_vec[~id] = 300 + T_ambient

for i in range(N_t):
    
    T_holder_inserted_avg = np.mean(T_holder[0:N_l_inserted])
    T_holder_ext_avg = np.mean(T_holder[N_l_inserted:])
    Q_i = h_steel_air*A_holder_inserted*(T_set_vec[i] - T_holder_inserted_avg) # heat exchange through convection
    q_i = Q_i/N_l_inserted/delta_vol
    Q_e = h_steel_air*A_holder_ext*(T_holder_ext_avg - T_ambient)
    q_e = Q_e/N_l_ext/delta_vol



    # Compute the temperature of the pin holder
    for jj in range(1,N_l-1):
        if jj <= N_l_inserted:
            qv = q_i
        else:
            qv = -q_e

        T_holder[jj] = sc.RK4_step_parametric(T_holder[jj], qv, T_dot_fun_holder, delta_t, parameters=vertcat(T_holder[jj-1], T_holder[jj+1]))

    # set dummy boundary conditions
    T_holder[-1] = T_holder[-2]
    T_holder[0] = T_holder[1]

In [132]:
print(delta_t)
print(delta_x**2/2/alpha_steel)
print(T_set_vec)

2.5205041008193803
8.775
[[800.]
 [800.]
 [800.]
 ...
 [600.]
 [600.]
 [600.]]


In [143]:
"""Plotting the results"""
%matplotlib qt5

l_vec = np.linspace(0, L, N_l)
plt.plot(l_vec, T_holder)

# print(r_num.shape)
# print(T_chamber.shape)
# print(T_chamber)

In [1]:
import casadi
print(casadi.__file__)


c:\Users\egrab\miniconda3\envs\researchCode\Lib\site-packages\casadi\__init__.py
